<a href="https://colab.research.google.com/github/slimani-meriem/w2vec/blob/main/01_dataset_preperation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

REPO_URL = "https://github.com/slimani-meriem/w2vec.git"
REPO_DIR = "/content/w2vec"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

sys.path.insert(0, f"{REPO_DIR}/src")

from data import *
from w2v import train_from_corpus
from eval_w2v import get_embeddings, evaluate, evaluate_runs

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 2), reused 4 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 424 bytes | 424.00 KiB/s, done.
From https://github.com/slimani-meriem/w2vec
   da39a5d..5101fb1  main       -> origin/main
Updating da39a5d..5101fb1
Fast-forward
 src/data.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


## Préparation du dataset

In [4]:
# Chemins
CORPUS       = f"{REPO_DIR}/data/Le_comte_de_Monte_Cristo.tok"
FILTERED     = f"{REPO_DIR}/data/filtered_text.txt"
DELETED      = f"{REPO_DIR}/data/deleted_words.txt"
DATASET_PATH = f"{REPO_DIR}/data/dataset.json"

# --- Hyperparamètres de préparation ---
L = 2          # taille du contexte gauche/droit
K = 10         # négatifs par exemple positif
MINC = 5       # occurrences minimales d'un mot
ALPHA = 0.75   # lissage de la distribution de tirage des négatifs
SEED = 0       # graine du tirage des négatifs

In [6]:
words_count, valid_words = filter_rare_words(CORPUS, FILTERED, minc=MINC,
                                             deleted_file=DELETED)

print(f"Mots distincts (corpus brut) : {len(words_count)}")
print(f"Mots conservés (count >= {MINC}) : {len(valid_words)}")

Mots distincts (corpus brut) : 13023
Mots conservés (count >= 5) : 3156


In [7]:
positive_samples = generate_positive_samples(FILTERED, L=L)

n_pairs = sum(len(ctx) for _, ctx in positive_samples)
print(f"Mots cibles : {len(positive_samples)}")
print(f"Paires (cible, contexte) positives : {n_pairs}")

Mots cibles : 233476
Paires (cible, contexte) positives : 876832


In [12]:
dataset = generate_negative_samples(positive_samples, words_count, valid_words,
                                        alpha=ALPHA, k=K, seed=SEED)
save_dataset(dataset, DATASET_PATH)
print("Dataset généré et sauvegardé.")

print(f"Taille du dataset : {len(dataset)} exemples")

Dataset généré et sauvegardé.
Taille du dataset : 876832 exemples


In [14]:
# La première phrase avec ses contextes
with open(CORPUS, "r", encoding="utf-8") as f:
    first_line = f.readline().strip()
f.close()
print(f"Première phrase avant filtrage :\n\"{first_line}\"\n")

with open(FILTERED, "r", encoding="utf-8") as f:
    first_line_filtered = f.readline().strip()
print(f"Première phrase filtrée :\n\"{first_line_filtered}\"\n")

print(f"{'Cible':<15} | Contexte")
print("-" * 50)
for target, context in positive_samples[:10]:
    print(f"{target:<15} | {context}")

# Un exemple complet
t, p, negs = dataset[0]
print(f"\nExemple 0 : cible={t!r}, positif={p!r}")
print(f"négatifs : {negs}")

Première phrase avant filtrage :
"<s> <s> le 24 février 1815 , la vigie de notre - dame de la garde signala le trois - mâts le pharaon , venant de smyrne , trieste et naples . </s> </s>"

Première phrase filtrée :
"<s> <s> le février 1815 , la de notre - dame de la garde le trois - le pharaon , venant de smyrne , et naples . </s> </s>"

Cible           | Contexte
--------------------------------------------------
<s>             | ['<s>', 'le']
<s>             | ['<s>', 'le', 'février']
le              | ['<s>', '<s>', 'février', '1815']
février         | ['<s>', 'le', '1815', ',']
1815            | ['le', 'février', ',', 'la']
,               | ['février', '1815', 'la', 'de']
la              | ['1815', ',', 'de', 'notre']
de              | [',', 'la', 'notre', '-']
notre           | ['la', 'de', '-', 'dame']
-               | ['de', 'notre', 'dame', 'de']

Exemple 0 : cible='<s>', positif='<s>'
négatifs : ['le', 'surprise', 'occasion', 'il', np.str_('bien'), 'coton', 'trouver', 'elle'